# Week 6–7 Practice Exercise: Solution Key

**Stationarity, Unit Root Tests, Differencing, AR/MA/ARMA**

This notebook implements the full pipeline on the healthcare time series: load → ADF → differencing → ACF/PACF → ARIMA fit → residual diagnostics and forecast evaluation.

## 1. Setup and data load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
df = pd.read_csv("data/healthcare_ts.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
y = df["value"]

print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("\nDescribe:")
print(y.describe())
y.head()

In [ ]:
y.plot(title="Daily hospital admissions (simulated)", xlabel="Date", ylabel="Value")
plt.tight_layout()
plt.show()

## 2. Unit root tests (ADF on raw series)

In [ ]:
adf_raw = adfuller(y, autolag="AIC")
print("Augmented Dickey-Fuller test (raw series):")
print("  Test statistic:", round(adf_raw[0], 4))
print("  p-value:", round(adf_raw[1], 4))
print("  Critical values:", adf_raw[4])
if adf_raw[1] > 0.05:
    print("  Conclusion: Do not reject H0 → series has a unit root (non-stationary).")
else:
    print("  Conclusion: Reject H0 → series is stationary.")

## 3. Differencing

In [ ]:
dy = y.diff().dropna()
dy.plot(title="First difference of daily admissions", xlabel="Date", ylabel="Difference")
plt.tight_layout()
plt.show()

In [ ]:
adf_diff = adfuller(dy, autolag="AIC")
print("Augmented Dickey-Fuller test (differenced series):")
print("  Test statistic:", round(adf_diff[0], 4))
print("  p-value:", round(adf_diff[1], 4))
if adf_diff[1] <= 0.05:
    print("  Conclusion: Reject H0 → differenced series is stationary. Use d = 1.")

## 4. ACF and PACF for model selection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(dy, lags=40, ax=axes[0], title="ACF of differenced series")
plot_pacf(dy, lags=40, ax=axes[1], title="PACF of differenced series", method="ywm")
plt.tight_layout()
plt.show()
print("Interpretation: PACF cuts off after lag 1 → suggest AR(1) on diff. ACF decays.")
print("Suggested order for original series: ARIMA(1, 1, 0). Also try (0,1,1) and (1,1,1).")

## 5. AR/MA/ARMA fit

In [ ]:
orders = [(1, 1, 0), (0, 1, 1), (1, 1, 1)]
results = {}
for order in orders:
    model = ARIMA(y, order=order)
    fit = model.fit()
    results[order] = fit
    print(f"ARIMA{order}: AIC = {fit.aic:.2f}, BIC = {fit.bic:.2f}")
    print(f"  Params: {fit.params.to_dict()}")
    print()

In [ ]:
best_order = min(results, key=lambda k: results[k].bic)
fit_best = results[best_order]
print(f"Preferred model by BIC: ARIMA{best_order}")
print(fit_best.summary())

## 6. Evaluation

### 6.1 Residual diagnostics

In [ ]:
resid = fit_best.resid
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(resid, lags=40, ax=axes[0], title="ACF of residuals")
resid.hist(ax=axes[1], bins=30, edgecolor="black", alpha=0.7)
axes[1].set_title("Histogram of residuals")
plt.tight_layout()
plt.show()

In [ ]:
lb = acorr_ljungbox(resid, lags=20, return_df=True)
print("Ljung-Box test (residuals):")
print(lb[["lb_stat", "lb_pvalue"]].head(10))
print("\nIf p-values are above 0.05, we do not reject white noise → residuals are consistent with WN.")

### 6.2 Forecast evaluation (train/test split)

In [ ]:
test_frac = 0.15
n_test = int(len(y) * test_frac)
train = y.iloc[:-n_test]
test = y.iloc[-n_test:]

model_f = ARIMA(train, order=best_order)
fit_f = model_f.fit()
forecast = fit_f.forecast(steps=n_test)

rmse = np.sqrt(np.mean((test.values - forecast.values) ** 2))
mae = np.mean(np.abs(test.values - forecast.values))
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")

In [ ]:
train.iloc[-100:].plot(label="Train (last 100)", legend=True)
test.plot(label="Actual test", legend=True)
pd.Series(forecast.values, index=test.index).plot(label="Forecast", legend=True, linestyle="--")
plt.title("Forecast vs actual (test set)")
plt.ylabel("Value")
plt.tight_layout()
plt.show()